In [1]:
from dotenv import load_dotenv
load_dotenv()

import pandas as pd
from google import genai

client = genai.Client()
MODEL = "gemini-3.5-flash-lite"

BASE = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data"
train = pd.read_csv(f"{BASE}/train.csv")

categories = sorted(train["category"].unique()) + ["out_of_scope"]
print(len(categories), "categories")

78 categories


In [2]:
from enum import Enum
from pydantic import BaseModel

Category = Enum("Category", {name: name for name in categories})

class Classification(BaseModel):
    category: Category

In [4]:
from google.genai import types

PROMPT_V1 = """You classify customer messages sent to a bank's support team.
Choose the single best category for the message.
If the message is not about banking, choose out_of_scope."""

def classify(text: str):
    response = client.models.generate_content(
        model=MODEL,
        contents=text,
        config=types.GenerateContentConfig(
            system_instruction=PROMPT_V1,
            response_mime_type="application/json",
            response_schema=Classification,
        ),
    )
    return response

In [5]:
tests = [
    "my card still hasn't arrived",
    "I was charged twice for the same coffee",
    "what is the exchange rate for euros?",
    "someone stole my wallet with my card in it",
    "tell me a joke",
]

for text in tests:
    response = classify(text)
    result = response.parsed   # already converted into a Classification object
    print(f"{text!r:50} → {result.category.value}   (tokens in: {response.usage_metadata.prompt_token_count})")

"my card still hasn't arrived"                     → card_arrival   (tokens in: 48)
'I was charged twice for the same coffee'          → transaction_charged_twice   (tokens in: 49)
'what is the exchange rate for euros?'             → exchange_rate   (tokens in: 49)
'someone stole my wallet with my card in it'       → lost_or_stolen_card   (tokens in: 50)
'tell me a joke'                                   → out_of_scope   (tokens in: 45)
